In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import cv2
import matplotlib.pyplot as plt

In [7]:
PROJECT_PATH = Path("../")

DATASET_PATH = PROJECT_PATH / "02_datasets" / "DUT_Anti_UAV"

MODEL_PATH = (
    PROJECT_PATH
    / "runs"
    / "detect"
    / "04_experiments"
    / "EXP001_YOLOv8s_baseline"
    / "weights"
    / "best.pt"
)

DATASET_PATH, MODEL_PATH

(WindowsPath('../02_datasets/DUT_Anti_UAV'),
 WindowsPath('../runs/detect/04_experiments/EXP001_YOLOv8s_baseline/weights/best.pt'))

In [8]:
test_labels = DATASET_PATH / "labels" / "test"

gt_data = []

for label_file in test_labels.glob("*.txt"):
    
    image_name = label_file.stem
    
    with open(label_file, "r") as f:
        lines = f.readlines()
    
    for line in lines:
        values = line.strip().split()
        
        cls = int(values[0])
        x = float(values[1])
        y = float(values[2])
        w = float(values[3])
        h = float(values[4])
        
        area = w * h
        
        gt_data.append({
            "image": image_name,
            "class": cls,
            "width": w,
            "height": h,
            "area": area
        })

gt_df = pd.DataFrame(gt_data)

gt_df.head()

,image,class,width,height,area
0,00001,0,0.053125,0.084722,0.004501
1,00002,0,0.048438,0.083333,0.004036
2,00003,0,0.225781,0.151389,0.034181
3,00004,0,0.089063,0.058333,0.005195
4,00005,0,0.017969,0.026389,0.000474


In [9]:
def size_category(area):
    
    if area < 0.001:
        return "Tiny"
    
    elif area < 0.01:
        return "Small"
    
    elif area < 0.1:
        return "Medium"
    
    else:
        return "Large"


gt_df["size_category"] = gt_df["area"].apply(size_category)

gt_df["size_category"].value_counts()

size_category
Tiny      1148
Small      555
Medium     467
Large       75
Name: count, dtype: int64

In [13]:
from ultralytics import YOLO
model = YOLO(MODEL_PATH)

results = model.val(
    data=str(DATASET_PATH / "data.yaml"),
    split="test"
)

Ultralytics 8.4.124  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 901.0680.4 MB/s, size: 80.8 KB)
val: Scanning D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\labels\test.cache... 2200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2200/2200  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 138/138 7.5it/s 18.5s0.1s
                   all       2200       2245      0.966      0.927      0.951      0.652
Speed: 1.1ms preprocess, 3.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\master\Master_Drone_Detection\notebooks\runs\detect\val


In [14]:
from ultralytics import YOLO
import pandas as pd

model = YOLO(MODEL_PATH)

test_images = list((DATASET_PATH / "Images" / "test").glob("*.jpg"))

len(test_images)

2200

In [15]:
predictions = []

for img_path in test_images:
    
    result = model.predict(
        source=str(img_path),
        conf=0.25,
        verbose=False,
        device=0
    )[0]
    
    boxes = result.boxes
    
    detected = len(boxes)
    
    predictions.append({
        "image": img_path.stem,
        "detections": detected,
        "confidence_mean": float(boxes.conf.mean()) if detected > 0 else 0
    })

pred_df = pd.DataFrame(predictions)

pred_df.head()

,image,detections,confidence_mean
0,00001,1,0.877906
1,00002,1,0.873486
2,00003,1,0.862339
3,00004,1,0.874388
4,00005,1,0.690354


In [16]:
merged_df = gt_df.merge(
    pred_df,
    on="image",
    how="left"
)

In [17]:
merged_df = gt_df.merge(
    pred_df,
    on="image",
    how="left"
)

merged_df.head()

,image,class,width,height,area,size_category,detections,confidence_mean
0,00001,0,0.053125,0.084722,0.004501,Small,1,0.877906
1,00002,0,0.048438,0.083333,0.004036,Small,1,0.873486
2,00003,0,0.225781,0.151389,0.034181,Medium,1,0.862339
3,00004,0,0.089063,0.058333,0.005195,Small,1,0.874388
4,00005,0,0.017969,0.026389,0.000474,Tiny,1,0.690354


In [18]:
merged_df["detected"] = merged_df["detections"] > 0

merged_df.head()

,image,class,width,height,area,size_category,detections,confidence_mean,detected
0,00001,0,0.053125,0.084722,0.004501,Small,1,0.877906,True
1,00002,0,0.048438,0.083333,0.004036,Small,1,0.873486,True
2,00003,0,0.225781,0.151389,0.034181,Medium,1,0.862339,True
3,00004,0,0.089063,0.058333,0.005195,Small,1,0.874388,True
4,00005,0,0.017969,0.026389,0.000474,Tiny,1,0.690354,True


In [19]:
size_detection = (
    merged_df
    .groupby("size_category")
    .agg(
        total_objects=("image", "count"),
        detected_objects=("detected", "sum")
    )
)

size_detection["detection_rate"] = (
    size_detection["detected_objects"] /
    size_detection["total_objects"]
)

size_detection

,total_objects,detected_objects,detection_rate
size_category,,,
Large,75,74,0.986667
Medium,467,454,0.972163
Small,555,535,0.963964
Tiny,1148,1096,0.954704


In [20]:
from ultralytics import YOLO
import pandas as pd

model = YOLO(MODEL_PATH)

pred_boxes = []

for img_path in test_images:

    result = model.predict(
        source=str(img_path),
        conf=0.25,
        verbose=False,
        device=0
    )[0]

    for box in result.boxes:

        xyxy = box.xyxy[0].cpu().numpy()

        pred_boxes.append({
            "image": img_path.stem,
            "x1": xyxy[0],
            "y1": xyxy[1],
            "x2": xyxy[2],
            "y2": xyxy[3],
            "confidence": float(box.conf[0])
        })

pred_boxes_df = pd.DataFrame(pred_boxes)

pred_boxes_df.head()

,image,x1,y1,x2,y2,confidence
0,00001,614.140015,541.470337,682.293091,602.050659,0.877906
1,00002,512.551147,226.074768,572.945312,284.010437,0.873486
2,00003,472.303223,77.581154,779.630249,190.478180,0.862339
3,00004,782.170227,67.874146,896.035828,118.388596,0.874388
4,00005,665.116089,353.410828,690.285034,373.290710,0.690354


In [21]:
from PIL import Image

gt_boxes = []

test_images_dir = DATASET_PATH / "Images" / "test"

for label_file in test_labels.glob("*.txt"):

    image_name = label_file.stem

    img_path = test_images_dir / f"{image_name}.jpg"

    img = Image.open(img_path)

    img_width, img_height = img.size

    with open(label_file, "r") as f:
        lines = f.readlines()

    for line in lines:

        values = line.strip().split()

        x_center = float(values[1])
        y_center = float(values[2])
        w = float(values[3])
        h = float(values[4])


        # convert normalized YOLO to pixels

        x1 = (x_center - w/2) * img_width
        y1 = (y_center - h/2) * img_height

        x2 = (x_center + w/2) * img_width
        y2 = (y_center + h/2) * img_height


        gt_boxes.append({
            "image": image_name,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2
        })


gt_boxes_df = pd.DataFrame(gt_boxes)

gt_boxes_df.head()

,image,x1,y1,x2,y2
0,00001,615.99936,540.00000,683.99936,600.99984
1,00002,512.99968,224.00028,575.00032,284.00004
2,00003,482.00000,78.00012,770.99968,187.00020
3,00004,782.00000,70.00020,896.00064,111.99996
4,00005,666.00000,352.99980,689.00032,371.99988


In [22]:
def calculate_iou(box1, box2):
    """
    box format:
    [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])


    intersection = max(0, x2 - x1) * max(0, y2 - y1)


    area1 = (
        (box1[2]-box1[0]) *
        (box1[3]-box1[1])
    )

    area2 = (
        (box2[2]-box2[0]) *
        (box2[3]-box2[1])
    )


    union = area1 + area2 - intersection


    if union == 0:
        return 0

    return intersection / union

In [23]:
gt_box = gt_boxes_df.iloc[0][
    ["x1","y1","x2","y2"]
].values


pred_box = pred_boxes_df.iloc[0][
    ["x1","y1","x2","y2"]
].values


iou = calculate_iou(gt_box, pred_box)

iou

np.float64(0.9113549851782781)

In [24]:
results_iou = []

for idx, gt_row in gt_boxes_df.iterrows():

    image_name = gt_row["image"]

    gt_box = [
        gt_row["x1"],
        gt_row["y1"],
        gt_row["x2"],
        gt_row["y2"]
    ]

    pred_rows = pred_boxes_df[
        pred_boxes_df["image"] == image_name
    ]

    if len(pred_rows) == 0:
        results_iou.append({
            "image": image_name,
            "iou": 0,
            "detected": False
        })

    else:

        best_iou = 0

        for _, pred_row in pred_rows.iterrows():

            pred_box = [
                pred_row["x1"],
                pred_row["y1"],
                pred_row["x2"],
                pred_row["y2"]
            ]

            iou = calculate_iou(gt_box, pred_box)

            best_iou = max(best_iou, iou)


        results_iou.append({
            "image": image_name,
            "iou": best_iou,
            "detected": best_iou >= 0.5
        })


iou_df = pd.DataFrame(results_iou)

iou_df.head()

,image,iou,detected
0,00001,0.911355,True
1,00002,0.926797,True
2,00003,0.907906,True
3,00004,0.830152,True
4,00005,0.838655,True


In [25]:
size_iou_df = gt_df.merge(
    iou_df,
    on="image"
)

size_iou_df.head()

,image,class,width,height,area,size_category,iou,detected
0,00001,0,0.053125,0.084722,0.004501,Small,0.911355,True
1,00002,0,0.048438,0.083333,0.004036,Small,0.926797,True
2,00003,0,0.225781,0.151389,0.034181,Medium,0.907906,True
3,00004,0,0.089063,0.058333,0.005195,Small,0.830152,True
4,00005,0,0.017969,0.026389,0.000474,Tiny,0.838655,True


In [26]:
iou_results = (
    size_iou_df
    .groupby("size_category")
    .agg(
        objects=("image","count"),
        correct=("detected","sum"),
        mean_iou=("iou","mean")
    )
)

iou_results["recall"] = (
    iou_results["correct"] /
    iou_results["objects"]
)

iou_results

,objects,correct,mean_iou,recall
size_category,,,,
Large,78,75,0.860673,0.961538
Medium,510,490,0.862826,0.960784
Small,623,595,0.848233,0.955056
Tiny,1148,1058,0.736335,0.921603


In [27]:
worst_cases = (
    size_iou_df
    .sort_values("iou")
    .head(20)
)

worst_cases[
    ["image","size_category","area","iou","detected"]
]

,image,size_category,area,iou,detected
2341,02183,Tiny,0.000236,0.0,False
696,00614,Tiny,0.000152,0.0,False
42,00043,Small,0.002398,0.0,False
50,00048,Small,0.008750,0.0,False
1002,00920,Tiny,0.000564,0.0,False
1037,00955,Tiny,0.000324,0.0,False
961,00879,Small,0.001283,0.0,False
984,00902,Tiny,0.000082,0.0,False
451,00377,Small,0.001296,0.0,False
1013,00931,Tiny,0.000685,0.0,False


In [32]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

worst_images = worst_cases["image"].values[:12]

img_dir = DATASET_PATH / "Images" / "test"

fig, axes = plt.subplots(3, 4, figsize=(15, 12))

for ax, img_name in zip(axes.flatten(), worst_images):

    img_path = img_dir / f"{img_name}.jpg"

    print(img_path, img_path.exists())

    img = cv2.imread(str(img_path))

    if img is None:
        ax.set_title(f"{img_name} NOT FOUND")
        ax.axis("off")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img)
    ax.set_title(img_name)
    ax.axis("off")


plt.tight_layout()

plt.savefig(
    "../06_results/EXP001_baseline/worst_cases.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

..\02_datasets\DUT_Anti_UAV\Images\test\02183.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00614.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00043.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00048.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00920.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00955.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00879.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00902.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00377.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00931.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\00872.jpg True
..\02_datasets\DUT_Anti_UAV\Images\test\02114.jpg True


<Figure size 640x480 with 0 Axes>

<Figure size 1500x1200 with 12 Axes>

In [34]:
import matplotlib.pyplot as plt
import cv2

sample_images = worst_cases["image"].values[:12]

fig, axes = plt.subplots(3,4, figsize=(15,12))


for ax, img_name in zip(axes.flatten(), sample_images):

    img_path = DATASET_PATH / "Images" / "test" / f"{img_name}.jpg"

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax.imshow(img)

    gt = gt_boxes_df[
        gt_boxes_df["image"] == img_name
    ]

    for _, box in gt.iterrows():

        ax.add_patch(
            plt.Rectangle(
                (box.x1, box.y1),
                box.x2-box.x1,
                box.y2-box.y1,
                fill=False,
                linewidth=2
            )
        )

    ax.set_title(img_name)
    ax.axis("off")

    plt.savefig(
    "../06_results/EXP001_baseline/BBOX.png",
    dpi=300,
    bbox_inches="tight"
)


plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 12 Axes>